In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

# ==========================================
# 1. LOAD PROCESSED DATA
# ==========================================

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("===== DATA LOADED =====")
print("Training features:", X_train.shape)
print("Testing features :", X_test.shape)
print("Training labels  :", y_train.shape)
print("Testing labels   :", y_test.shape)

# ==========================================
# 2. CREATE RANDOM FOREST
# ==========================================

model = RandomForestClassifier(
    n_estimators=150,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ==========================================
# 3. TRAIN MODEL
# ==========================================

print("\n===== TRAINING RANDOM FOREST =====")

model.fit(X_train, y_train)

print("Training complete!")

# ==========================================
# 4. PREDICTION
# ==========================================

y_pred = model.predict(X_test)

# ==========================================
# 5. EVALUATION
# ==========================================

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("\n===== RANDOM FOREST RESULTS =====")

print("\nMacro F1 Score:")
print(round(macro_f1, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Normal",
            "Suspect",
            "Pathological"
        ]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ==========================================
# 6. FEATURE IMPORTANCE
# ==========================================

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\n===== TOP 15 IMPORTANT FEATURES =====")
print(importance.head(15).to_string(index=False))

# ==========================================
# 7. SAVE MODEL
# ==========================================

os.makedirs("../models", exist_ok=True)

joblib.dump(
    model,
    "../models/random_forest.pkl"
)

# ==========================================
# 8. SAVE RESULTS
# ==========================================

os.makedirs("../results", exist_ok=True)

importance.to_csv(
    "../results/random_forest_feature_importance.csv",
    index=False
)

print("\n===== SAVED =====")
print("Model: models/random_forest.pkl")
print("Feature importance: results/random_forest_feature_importance.csv")

===== DATA LOADED =====
Training features: (1692, 41)
Testing features : (424, 41)
Training labels  : (1692,)
Testing labels   : (424,)

===== TRAINING RANDOM FOREST =====
Training complete!

===== RANDOM FOREST RESULTS =====

Macro F1 Score:
0.9932

Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00       330
     Suspect       1.00      0.97      0.98        59
Pathological       1.00      1.00      1.00        35

    accuracy                           1.00       424
   macro avg       1.00      0.99      0.99       424
weighted avg       1.00      1.00      1.00       424


Confusion Matrix:
[[330   0   0]
 [  2  57   0]
 [  0   0  35]]

===== TOP 15 IMPORTANT FEATURES =====
Feature  Importance
  CLASS    0.197235
   SUSP    0.105675
     LD    0.075008
     FS    0.068272
   ASTV    0.057541
   Mean    0.044444
   ALTV    0.042988
 Median    0.042411
      E    0.040466
     AC    0.034264
   Mode    0.029051
 

In [3]:
print("===== TOP 15 FEATURES =====")
print(importance.head(15).to_string(index=False))

===== TOP 15 FEATURES =====
Feature  Importance
  CLASS    0.197235
   SUSP    0.105675
     LD    0.075008
     FS    0.068272
   ASTV    0.057541
   Mean    0.044444
   ALTV    0.042988
 Median    0.042411
      E    0.040466
     AC    0.034264
   Mode    0.029051
   MSTV    0.027017
   AC.1    0.025455
   DP.1    0.022700
     DP    0.019179
